In [2]:
import fitz 
import re
import json
import pandas as pd
from pathlib import Path

RAW_DIR = Path("data/raw")
PROCESSED_DIR = Path("data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

CH4_PATH = RAW_DIR / "iesc104.pdf"
CH6_PATH = RAW_DIR / "iesc106.pdf"

print("Setup complete ")
print(f"Ch4 exists: {CH4_PATH.exists()}")
print(f"Ch6 exists: {CH6_PATH.exists()}")

Setup complete 
Ch4 exists: True
Ch6 exists: True


In [3]:
#PDF Extraction form the chapters in the raw
def extract_text_from_pdf(pdf_path):
    doc = fitz.open(pdf_path)
    pages = []
    for page_num, page in enumerate(doc, start=1):
        text = page.get_text()
        pages.append({
            "page": page_num,
            "text": text.strip()
        })
    doc.close()
    return pages

ch4_pages = extract_text_from_pdf(CH4_PATH)
ch6_pages = extract_text_from_pdf(CH6_PATH)

print(f"Ch4 — Pages extracted: {len(ch4_pages)}")
print(f"Ch6 — Pages extracted: {len(ch6_pages)}")

# Quick sanity check — print first 300 chars of page 2
print("\n── Ch4 Page 2 preview ──")
print(ch4_pages[1]["text"][:300])

print("\n── Ch6 Page 2 preview ──")
print(ch6_pages[1]["text"][:300])

Ch4 — Pages extracted: 24
Ch6 — Pages extracted: 22

── Ch4 Page 2 preview ──
Describing Motion Around Us
49
about some more physical quantities, such as displacement, average 
velocity and average acceleration. You will also learn to describe motion 
not only in words, but also with numbers, equations and graphs.
4.1 Motion in a Straight Line
You have learnt that when an obj

── Ch6 Page 2 preview ──
How Forces Affect Motion
95
Fig. 6.3: (a) Pushing a box 
kept on table or floor
Force by hand
Force of friction
Fig. 6.2: Measuring the 
weight of an object using a 
spring balance
While learning about force earlier, did you notice that whenever any 
type of force acting on an object was described, 


In [4]:
#Content Type Classification 
def classify_content_type(text):
    text_lower = text.lower()
    
    if re.search(r'example\s+\d+[\.:]\s*|answer\s*:', text_lower):
        return "worked_example"
    
    if re.search(r'revise,\s*reflect|pause and ponder|journey beyond|at a glance', text_lower):
        return "end_of_chapter"
    
    if re.search(r'activity\s+\d+[\.:]\s*', text_lower):
        return "activity"
    
    return "concept"

def split_into_sections(pages, chapter_name):
    sections = []
    current_section = "Introduction"
    
    for page_data in pages:
        text = page_data["text"]
        page_num = page_data["page"]
        
        # Detect section headings like 4.1, 4.2, 6.1 etc.
        heading_match = re.search(r'\n(\d+\.\d+[\.\d]*\s+[A-Z][^\n]{5,60})', text)
        if heading_match:
            current_section = heading_match.group(1).strip()
        
        content_type = classify_content_type(text)
        
        sections.append({
            "chapter": chapter_name,
            "page": page_num,
            "section": current_section,
            "content_type": content_type,
            "text": text
        })
    
    return sections

ch4_sections = split_into_sections(ch4_pages, "Chapter 4 - Describing Motion Around Us")
ch6_sections = split_into_sections(ch6_pages, "Chapter 6 - How Forces Affect Motion")

all_sections = ch4_sections + ch6_sections

df = pd.DataFrame(all_sections)
print("Content type distribution:")
print(df["content_type"].value_counts())
print(f"\nTotal pages classified: {len(all_sections)}")
print("\nSample section labels (Ch4):")
for s in ch4_sections[:5]:
    print(f"  Page {s['page']} | {s['content_type']:15} | {s['section'][:50]}")

Content type distribution:
content_type
concept           14
activity          13
worked_example    12
end_of_chapter     7
Name: count, dtype: int64

Total pages classified: 46

Sample section labels (Ch4):
  Page 1 | concept         | Introduction
  Page 2 | concept         | 4.1 Motion in a Straight Line
  Page 3 | concept         | 4.1.2 Distance travelled and displacement
  Page 4 | end_of_chapter  | 4.1.2 Distance travelled and displacement
  Page 5 | worked_example  | 4.1.3 Average speed and average velocity


In [5]:
#Tokenizer Comparison
from transformers import AutoTokenizer

# i have used these three tokenizer 
gpt2_tokenizer    = AutoTokenizer.from_pretrained("gpt2")
bert_tokenizer    = AutoTokenizer.from_pretrained("bert-base-uncased")
t5_tokenizer      = AutoTokenizer.from_pretrained("t5-small")

# 5 representative passages from our corpus
passages = [
    "Displacement is the net change in the position of an object between the two given instants of time.",
    
    "The average velocity of an object in a time interval is given by v_av = s/t, where s is displacement and t is time interval. The SI unit is metre per second (m/s).",
    
    "The kinematic equations for motion in a straight line with constant acceleration are: v = u + at, s = ut + (1/2)at², and v² = u² + 2as.",
    
    "Newton's second law of motion states: when a net force acts on an object, the object accelerates in the direction of the net force. Mathematically, F = ma.",
    
    "When brakes are applied to a moving vehicle, it moves some distance before coming to a stop. The distance depends upon the velocity, road surface, and braking capacity of the vehicle."
]

results = []
print(f"{'Passage':<6} {'GPT2-BPE':>10} {'BERT-WP':>10} {'T5-SP':>10}  Disagreement Example")
print("-" * 80)

for i, passage in enumerate(passages, 1):
    gpt2_tokens = gpt2_tokenizer.tokenize(passage)
    bert_tokens = bert_tokenizer.tokenize(passage)
    t5_tokens   = t5_tokenizer.tokenize(passage)
    
    g, b, t = len(gpt2_tokens), len(bert_tokens), len(t5_tokens)
    
    sci_words = ["displacement", "acceleration", "kinematic", "velocity", "v²"]
    disagreement = ""
    for word in sci_words:
        if word.lower() in passage.lower():
            gw = [tok for tok in gpt2_tokens if word[:4].lower() in tok.lower()]
            bw = [tok for tok in bert_tokens if word[:4].lower() in tok.lower()]
            if gw or bw:
                disagreement = f"'{word}' → GPT2:{gw[:2]} BERT:{bw[:2]}"
                break
    
    print(f"P{i:<5} {g:>10} {b:>10} {t:>10}  {disagreement[:55]}")
    results.append({"passage": i, "gpt2": g, "bert": b, "t5": t})

print("\n── Detailed token boundaries for Passage 3 (formula-heavy) ──")
p3 = passages[2]
print(f"\nGPT2 tokens: {gpt2_tokenizer.tokenize(p3)}")
print(f"\nBERT tokens: {bert_tokenizer.tokenize(p3)}")
print(f"\nT5   tokens: {t5_tokenizer.tokenize(p3)}")

Passage   GPT2-BPE    BERT-WP      T5-SP  Disagreement Example
--------------------------------------------------------------------------------
P1             22         20         22  'displacement' → GPT2:[] BERT:['displacement']
P2             43         44         50  'displacement' → GPT2:['Ġdisplacement'] BERT:['displace
P3             44         44         52  'acceleration' → GPT2:['Ġacceleration'] BERT:['accelera
P4             37         37         40  
P5             36         36         40  'velocity' → GPT2:['Ġvelocity'] BERT:['velocity']

── Detailed token boundaries for Passage 3 (formula-heavy) ──

GPT2 tokens: ['The', 'Ġk', 'inem', 'atic', 'Ġequations', 'Ġfor', 'Ġmotion', 'Ġin', 'Ġa', 'Ġstraight', 'Ġline', 'Ġwith', 'Ġconstant', 'Ġacceleration', 'Ġare', ':', 'Ġv', 'Ġ=', 'Ġu', 'Ġ+', 'Ġat', ',', 'Ġs', 'Ġ=', 'Ġut', 'Ġ+', 'Ġ(', '1', '/', '2', ')', 'at', 'Â²', ',', 'Ġand', 'Ġv', 'Â²', 'Ġ=', 'Ġu', 'Â²', 'Ġ+', 'Ġ2', 'as', '.']

BERT tokens: ['the', 'kin', '##ema', '##tic', 'e

In [6]:
# Chunking Strategy, started with chunk_size=300
def chunk_sections(sections, chunk_size=300, overlap=50):
    """
    Chunking strategy:
    - concept pages: 300 tokens, 50 overlap
    - worked_example: kept whole (problem + solution must stay together)
    - end_of_chapter: split into individual questions
    - activity: kept whole
    
    Tokenizer: BERT WordPiece (used consistently for both index and query)
    """
    chunks = []
    chunk_id = 0
    words_per_token = 0.75  

    for section in sections:
        text = section["text"]
        ctype = section["content_type"]
        
        # Clean text
        text = re.sub(r'\s+', ' ', text).strip()
        text = re.sub(r'Chapter-\d+\.indd.*', '', text).strip()
        
        if ctype == "worked_example":
            chunks.append({
                "chunk_id": chunk_id,
                "chapter": section["chapter"],
                "section": section["section"],
                "content_type": ctype,
                "page": section["page"],
                "text": text
            })
            chunk_id += 1

        elif ctype == "end_of_chapter":
            questions = re.split(r'(?=\n?\d+\.\s+[A-Z])', text)
            for q in questions:
                q = q.strip()
                if len(q) > 50:
                    chunks.append({
                        "chunk_id": chunk_id,
                        "chapter": section["chapter"],
                        "section": section["section"],
                        "content_type": ctype,
                        "page": section["page"],
                        "text": q
                    })
                    chunk_id += 1

        else:
            words = text.split()
            step = int(chunk_size * words_per_token)
            window = int((chunk_size + overlap) * words_per_token)
            
            if len(words) <= window:
                chunks.append({
                    "chunk_id": chunk_id,
                    "chapter": section["chapter"],
                    "section": section["section"],
                    "content_type": ctype,
                    "page": section["page"],
                    "text": text
                })
                chunk_id += 1
            else:
                for start in range(0, len(words), step):
                    chunk_text = " ".join(words[start:start + window])
                    if len(chunk_text) > 100:
                        chunks.append({
                            "chunk_id": chunk_id,
                            "chapter": section["chapter"],
                            "section": section["section"],
                            "content_type": ctype,
                            "page": section["page"],
                            "text": chunk_text
                        })
                        chunk_id += 1

    return chunks

chunks = chunk_sections(all_sections, chunk_size=300, overlap=50)

# Save to disk, find it in the processed folder 
with open(PROCESSED_DIR / "chunks.json", "w") as f:
    json.dump(chunks, f, indent=2)

# Summary
chunks_df = pd.DataFrame(chunks)
print(f"Total chunks created: {len(chunks)}")
print(f"\nChunks by content type:")
print(chunks_df["content_type"].value_counts())
print(f"\nChunks by chapter:")
print(chunks_df["chapter"].value_counts())
print(f"\nAvg chunk length (words): {chunks_df['text'].apply(lambda x: len(x.split())).mean():.0f}")
print(f"\nSample chunk (worked_example):")
ex = chunks_df[chunks_df.content_type == "worked_example"].iloc[0]
print(f"  Section: {ex['section']}")
print(f"  Preview: {ex['text'][:200]}")

Total chunks created: 101

Chunks by content type:
content_type
activity          36
concept           32
end_of_chapter    21
worked_example    12
Name: count, dtype: int64

Chunks by chapter:
chapter
Chapter 6 - How Forces Affect Motion       51
Chapter 4 - Describing Motion Around Us    50
Name: count, dtype: int64

Avg chunk length (words): 225

Sample chunk (worked_example):
  Section: 4.1.3 Average speed and average velocity
  Preview: Exploration|Grade 9 52 Motion, i.e., a change in the position of an object, can be described in terms of the total distance travelled by the object and its displacement. But how can you describe how f


In [8]:
# ── Cell 5b: Chunk Cleaning + BM25 + Retrieve ────────────────────────
from rank_bm25 import BM25Okapi

def simple_tokenize(text):
    text = text.lower()
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    return text.split()

def clean_chunk_text(text):
    text = re.sub(r'\d*\s*Exploration\|Grade\s*9\s*\d*', '', text)
    text = re.sub(r'Fig\.?\s*\d+\.\d+[a-z]?\s*:?[^\n]*', '', text)
    text = re.sub(r'Chapter-\d+\.indd.*', '', text)
    text = re.sub(r'^\s*\d{1,3}\s*$', '', text, flags=re.MULTILINE)
    text = re.sub(r'Table\s+\d+\.\d+\s*:?[^\n]*', '', text)
    text = re.sub(r'Grade\s+\d+\s*Curiosity\s*Chapter\s*\d+', '', text)
    text = re.sub(r'\n{3,}', '\n\n', text)
    text = re.sub(r'\s{2,}', ' ', text)
    return text.strip()

# Clean chunks
for chunk in chunks:
    chunk["text"] = clean_chunk_text(chunk["text"])

chunks = [c for c in chunks if len(c["text"].split()) > 30]
print(f"Chunks after cleaning: {len(chunks)}")

# Save
with open(PROCESSED_DIR / "chunks.json", "w") as f:
    json.dump(chunks, f, indent=2)

# Build BM25
corpus_tokens = [simple_tokenize(chunk["text"]) for chunk in chunks]
bm25 = BM25Okapi(corpus_tokens)
print("BM25 index rebuilt on clean chunks ✓")

# Define retrieve
def retrieve(query, top_k=3, boost_concept=True):
    query_tokens = simple_tokenize(query)
    scores = bm25.get_scores(query_tokens)
    
    if boost_concept:
        for i, chunk in enumerate(chunks):
            if chunk["content_type"] == "concept":
                scores[i] *= 1.3
            elif chunk["content_type"] == "worked_example":
                scores[i] *= 1.2
            elif chunk["content_type"] == "end_of_chapter":
                scores[i] *= 0.8
    
    top_indices = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:top_k]
    
    results = []
    for idx in top_indices:
        chunk = chunks[idx].copy()
        chunk["bm25_score"] = round(scores[idx], 4)
        results.append(chunk)
    return results

# Test
test_queries = [
    "What is displacement and how is it different from distance?",
    "State Newton's second law of motion with formula",
    "What happens to velocity in uniform circular motion?"
]

for query in test_queries:
    print(f"\n{'='*60}")
    print(f"Query: {query}")
    print(f"{'='*60}")
    results = retrieve(query, top_k=3)
    for i, r in enumerate(results, 1):
        print(f"\n  Rank {i} | Score: {r['bm25_score']} | {r['content_type']} | Page {r['page']}")
        print(f"  Section: {r['section']}")
        print(f"  Preview: {r['text'][:150]}...")

Chunks after cleaning: 71
BM25 index rebuilt on clean chunks ✓

Query: What is displacement and how is it different from distance?

  Rank 1 | Score: 10.7321 | concept | Page 23
  Section: 4.4 Motion in a Plane
  Preview: by drawing a velocity-time graph for its motion. 15. Two cars A and B start moving with a constant acceleration from rest in a straight line. Car A at...

  Rank 2 | Score: 9.8869 | concept | Page 16
  Section: 4.3 Kinematic Equations for Motion in a Straight
  Preview: Describing Motion Around Us 63 This is the displacement of the car from the origin in 6 seconds. You have found that the area enclosed by the velocity...

  Rank 3 | Score: 9.3237 | worked_example | Page 5
  Section: 4.1.3 Average speed and average velocity
  Preview: Motion, i.e., a change in the position of an object, can be described in terms of the total distance travelled by the object and its displacement. But...

Query: State Newton's second law of motion with formula

  Rank 1 | Score: 10.8409 

In [9]:
#Improved Retriever
def retrieve(query, top_k=3, boost_concept=True):
    query_tokens = simple_tokenize(query)
    scores = bm25.get_scores(query_tokens)
    
    if boost_concept:
        for i, chunk in enumerate(chunks):
            if chunk["content_type"] == "concept":
                scores[i] *= 1.3
            elif chunk["content_type"] == "worked_example":
                scores[i] *= 1.2
            elif chunk["content_type"] == "end_of_chapter":
                scores[i] *= 0.8  # slight penalty — these are questions not answers
    
    top_indices = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:top_k]
    
    results = []
    for idx in top_indices:
        chunk = chunks[idx].copy()
        chunk["bm25_score"] = round(scores[idx], 4)
        results.append(chunk)
    return results

test_queries = [
    "What is displacement and how is it different from distance?",
    "State Newton's second law of motion with formula",
    "What happens to velocity in uniform circular motion?"
]

for query in test_queries:
    print(f"\n{'='*60}")
    print(f"Query: {query}")
    print(f"{'='*60}")
    results = retrieve(query, top_k=3)
    for i, r in enumerate(results, 1):
        print(f"\n  Rank {i} | Score: {r['bm25_score']} | {r['content_type']} | Page {r['page']}")
        print(f"  Section: {r['section']}")
        print(f"  Preview: {r['text'][:200]}...")


Query: What is displacement and how is it different from distance?

  Rank 1 | Score: 10.7321 | concept | Page 23
  Section: 4.4 Motion in a Plane
  Preview: by drawing a velocity-time graph for its motion. 15. Two cars A and B start moving with a constant acceleration from rest in a straight line. Car A attains a velocity of 5 m s–1 in 5 s. Car B attains ...

  Rank 2 | Score: 9.8869 | concept | Page 16
  Section: 4.3 Kinematic Equations for Motion in a Straight
  Preview: Describing Motion Around Us 63 This is the displacement of the car from the origin in 6 seconds. You have found that the area enclosed by the velocity-time graph and the time axis for a desired time i...

  Rank 3 | Score: 9.3237 | worked_example | Page 5
  Section: 4.1.3 Average speed and average velocity
  Preview: Motion, i.e., a change in the position of an object, can be described in terms of the total distance travelled by the object and its displacement. But how can you describe how fast or slow an object i.

In [ ]:
pip install groq --break-system-packages

Note: you may need to restart the kernel to use updated packages.


In [10]:
import os
from groq import Groq
client = Groq(api_key=os.getenv("GROQ_API_KEY"))
response = client.chat.completions.create(
    model="llama-3.1-8b-instant",
    messages=[{"role": "user", "content": "Say hello"}]
)
print(response.choices[0].message.content)

Hello. Is there something I can help you with or would you like to chat?


In [11]:
# ── Cell 7: Generation with Groq ─────────────────────────────────────
import os
from groq import Groq
from dotenv import load_dotenv

load_dotenv()
groq_client = Groq(api_key=os.getenv("GROQ_API_KEY"))

GROUNDING_PROMPT = """You are a science study assistant for Class 9 students using NCERT textbooks.

You will be given CONTEXT extracted from NCERT Class 9 Science chapters.
Your job is to answer the student's question STRICTLY using only the provided context.

STRICT RULES:
1. If the answer is present in the context, answer clearly and simply for a Class 9 student.
2. If the answer is NOT in the context, respond with exactly: "I'm sorry, this topic is not covered in the provided chapter content. Please refer to your teacher or other chapters."
3. Do NOT use any outside knowledge, even if you know the answer.
4. Do NOT make up formulas, definitions, or examples not present in the context.
5. Keep answers concise — 3 to 5 sentences maximum.
6. If a formula is in the context, include it in your answer.

CONTEXT:
{context}

STUDENT QUESTION:
{question}

ANSWER:"""

def format_context(retrieved_chunks):
    parts = []
    for i, chunk in enumerate(retrieved_chunks, 1):
        parts.append(
            f"[Source {i} | {chunk['chapter']} | {chunk['section']} | Page {chunk['page']}]\n"
            f"{chunk['text']}"
        )
    return "\n\n".join(parts)

def answer(question, top_k=3, temperature=0.0):
    retrieved = retrieve(question, top_k=top_k)
    context = format_context(retrieved)
    prompt = GROUNDING_PROMPT.format(context=context, question=question)
    
    response = groq_client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": prompt}],
        temperature=temperature,
        max_tokens=1024
    )
    
    return {
        "question": question,
        "answer": response.choices[0].message.content.strip(),
        "retrieved_chunks": retrieved,
        "context_used": context
    }

# ── Smoke test ────────────────────────────────────────────────────────
test_q = "What is Newton's second law of motion?"
result = answer(test_q)
print(f"Question: {result['question']}")
print(f"\nAnswer:\n{result['answer']}")
print(f"\nRetrieved from:")
for c in result['retrieved_chunks']:
    print(f"  - {c['section']} | Page {c['page']} | Score {c['bm25_score']}")

Question: What is Newton's second law of motion?

Answer:
Newton's second law of motion states that the rate of change of momentum of an object is proportional to the net force and takes place in the direction in which the net force acts.

Retrieved from:
  - 6.5 Newton’s Second Law of Motion | Page 11 | Score 12.2819
  - 6.7 Forces Acting on a System of Objects | Page 18 | Score 11.096
  - 6.1 The Concept of Force | Page 1 | Score 10.2997


In [12]:
#Evaluation Set 
evaluation_questions = [
    # Taken from the textbook (10 questions)
    {
        "id": 1,
        "question": "What is displacement?",
        "expected_topic": "displacement definition",
        "category": "direct",
        "chapter": "Ch4"
    },
    {
        "id": 2,
        "question": "What are the three kinematic equations of motion?",
        "expected_topic": "kinematic equations v=u+at",
        "category": "direct",
        "chapter": "Ch4"
    },
    {
        "id": 3,
        "question": "What is uniform circular motion?",
        "expected_topic": "circular motion constant speed",
        "category": "direct",
        "chapter": "Ch4"
    },
    {
        "id": 4,
        "question": "What is average acceleration?",
        "expected_topic": "average acceleration change in velocity",
        "category": "direct",
        "chapter": "Ch4"
    },
    {
        "id": 5,
        "question": "State Newton's first law of motion.",
        "expected_topic": "object at rest remains at rest",
        "category": "direct",
        "chapter": "Ch6"
    },
    {
        "id": 6,
        "question": "State Newton's second law of motion.",
        "expected_topic": "F=ma net force acceleration",
        "category": "direct",
        "chapter": "Ch6"
    },
    {
        "id": 7,
        "question": "State Newton's third law of motion.",
        "expected_topic": "equal and opposite force",
        "category": "direct",
        "chapter": "Ch6"
    },
    {
        "id": 8,
        "question": "What is the SI unit of force and how is it defined?",
        "expected_topic": "newton 1kg 1ms-2",
        "category": "direct",
        "chapter": "Ch6"
    },
    {
        "id": 9,
        "question": "What is the force of friction and in which direction does it act?",
        "expected_topic": "friction opposite direction of motion",
        "category": "direct",
        "chapter": "Ch6"
    },
    {
        "id": 10,
        "question": "How do you calculate average speed?",
        "expected_topic": "total distance divided by time",
        "category": "direct",
        "chapter": "Ch4"
    },

    {
        "id": 11,
        "question": "Why does a fielder pull hands back while catching a fast cricket ball?",
        "expected_topic": "Newton second law time force injury",
        "category": "paraphrased",
        "chapter": "Ch6"
    },
    {
        "id": 12,
        "question": "If an object moves in a circle at constant speed is it accelerating?",
        "expected_topic": "direction changes so acceleration exists",
        "category": "paraphrased",
        "chapter": "Ch4"
    },
    {
        "id": 13,
        "question": "What decides how far a car travels after brakes are applied?",
        "expected_topic": "velocity road surface braking capacity",
        "category": "paraphrased",
        "chapter": "Ch4"
    },

    {
        "id": 14,
        "question": "What is photosynthesis?",
        "expected_topic": "REFUSAL - biology not in corpus",
        "category": "out_of_scope",
        "chapter": "None"
    },
    {
        "id": 15,
        "question": "Explain the water cycle.",
        "expected_topic": "REFUSAL - geography not in corpus",
        "category": "out_of_scope",
        "chapter": "None"
    },
    {
        "id": 16,
        "question": "What is quantum entanglement?",
        "expected_topic": "REFUSAL - advanced physics not in corpus",
        "category": "out_of_scope",
        "chapter": "None"
    },
    {
        "id": 17,
        "question": "Who is the Prime Minister of India?",
        "expected_topic": "REFUSAL - not science content",
        "category": "out_of_scope",
        "chapter": "None"
    },
    {
        "id": 18,
        "question": "Explain Newton's law of gravitation and calculate gravitational force between two planets.",
        "expected_topic": "REFUSAL - gravitation chapter not in corpus",
        "category": "out_of_scope",
        "chapter": "None"
    },
]

print(f"Total evaluation questions: {len(evaluation_questions)}")
print(f"Direct:       {sum(1 for q in evaluation_questions if q['category'] == 'direct')}")
print(f"Paraphrased:  {sum(1 for q in evaluation_questions if q['category'] == 'paraphrased')}")
print(f"Out of scope: {sum(1 for q in evaluation_questions if q['category'] == 'out_of_scope')}")

Total evaluation questions: 18
Direct:       10
Paraphrased:  3
Out of scope: 5


In [14]:
#Cell 9: Run Evaluation
import time

def evaluate_answer(question_data, result):
    """
    Manual scoring guide — you review each answer after running.
    Returns a dict with placeholders you fill in.
    """
    return {
        "id": question_data["id"],
        "question": question_data["question"],
        "category": question_data["category"],
        "chapter": question_data["chapter"],
        "expected_topic": question_data["expected_topic"],
        "answer": result["answer"],
        "top_chunk_section": result["retrieved_chunks"][0]["section"],
        "top_chunk_page": result["retrieved_chunks"][0]["page"],
        "top_chunk_type": result["retrieved_chunks"][0]["content_type"],
        "correctness": "?",       # yes / partial / no
        "grounded": "?",          # yes / no
        "refusal_appropriate": "?" if question_data["category"] != "out_of_scope" else "?"
    }

# Runing all questions
print("Running evaluation... (this may take ~1 min due to API calls)\n")
eval_results = []

for q in evaluation_questions:
    try:
        result = answer(q["question"], top_k=3, temperature=0.0)
        row = evaluate_answer(q, result)
        eval_results.append(row)
        
        print(f"[Q{q['id']:02d}] [{q['category']:12}] {q['question'][:55]}")
        print(f"       Answer: {result['answer'][:120]}...")
        print(f"       Top chunk: {result['retrieved_chunks'][0]['section'][:50]}")
        print()
        
        time.sleep(2)  
        
    except Exception as e:
        print(f"[Q{q['id']:02d}] ERROR: {e}")
        eval_results.append({
            "id": q["id"], "question": q["question"],
            "category": q["category"], "answer": f"ERROR: {e}",
            "correctness": "no", "grounded": "no"
        })

print(f"\nEvaluation complete. {len(eval_results)} questions processed.")

eval_df = pd.DataFrame(eval_results)
eval_df.to_csv("evaluation/evaluation_results_raw.csv", index=False)
print("Saved to evaluation/evaluation_results_raw.csv")

Running evaluation... (this may take ~1 min due to API calls)

[Q01] [direct      ] What is displacement?
       Answer: Displacement is the change in the position of an object from its initial position to its final position. It is a vector ...
       Top chunk: 4.4 Motion in a Plane

[Q02] [direct      ] What are the three kinematic equations of motion?
       Answer: The three kinematic equations of motion are:

1. at = v - u 
2. v = u + at 
3. v^2 = u^2 + 2as...
       Top chunk: 4.3 Kinematic Equations for Motion in a Straight

[Q03] [direct      ] What is uniform circular motion?
       Answer: When an object moves in a circular path with constant (uniform) speed, its motion is called uniform circular motion....
       Top chunk: 4.4 Motion in a Plane

[Q04] [direct      ] What is average acceleration?
       Answer: The average acceleration of an object over a time interval is the change in its velocity divided by the time interval. T...
       Top chunk: 4.1.4 Average accelerati

In [15]:
# Read from saved CSV instead
eval_df = pd.read_csv("evaluation/evaluation_results_raw.csv")
for _, row in eval_df.iterrows():
    print(f"{'─'*60}")
    print(f"Q{row['id']} | {row['category']} | {row['question']}")
    print(f"ANSWER: {row['answer']}")
    print(f"CHUNK: {row['top_chunk_section']}")
    print()

────────────────────────────────────────────────────────────
Q1 | direct | What is displacement?
ANSWER: Displacement is the change in the position of an object from its initial position to its final position. It is a vector quantity, which means it has both magnitude and direction.
CHUNK: 4.4 Motion in a Plane

────────────────────────────────────────────────────────────
Q2 | direct | What are the three kinematic equations of motion?
ANSWER: The three kinematic equations of motion are:

1. at = v - u 
2. v = u + at 
3. v^2 = u^2 + 2as
CHUNK: 4.3 Kinematic Equations for Motion in a Straight

────────────────────────────────────────────────────────────
Q3 | direct | What is uniform circular motion?
ANSWER: When an object moves in a circular path with constant (uniform) speed, its motion is called uniform circular motion.
CHUNK: 4.4 Motion in a Plane

────────────────────────────────────────────────────────────
Q4 | direct | What is average acceleration?
ANSWER: The average acceleration 

In [16]:
#Cell 10: Save Scored Results
scores = {
    1:  ("partial", "yes", "n/a"),
    2:  ("partial", "yes", "n/a"),
    3:  ("yes", "yes", "n/a"),
    4:  ("yes", "yes", "n/a"),
    5:  ("yes", "yes", "n/a"),
    6:  ("yes", "yes", "n/a"),
    7:  ("yes", "yes", "n/a"),
    8:  ("yes", "yes", "n/a"),
    9:  ("yes", "yes", "n/a"),
    10: ("yes", "yes", "n/a"),
    11: ("yes", "yes", "n/a"),
    12: ("yes", "yes", "n/a"),
    13: ("yes", "yes", "n/a"),
    14: ("n/a", "n/a", "yes"),
    15: ("n/a", "n/a", "yes"),
    16: ("n/a", "n/a", "yes"),
    17: ("n/a", "n/a", "yes"),
    18: ("n/a", "n/a", "yes"),
}

eval_df = pd.read_csv("evaluation/evaluation_results_raw.csv")
eval_df["correctness"] = eval_df["id"].map(lambda x: scores[x][0])
eval_df["grounded"] = eval_df["id"].map(lambda x: scores[x][1])
eval_df["refusal_appropriate"] = eval_df["id"].map(lambda x: scores[x][2])

eval_df.to_csv("evaluation/evaluation_results.csv", index=False)

# Print summary
correct = sum(1 for v in scores.values() if v[0] == "yes")
partial = sum(1 for v in scores.values() if v[0] == "partial")
refusals = sum(1 for v in scores.values() if v[2] == "yes")

print(f"Total questions: 18")
print(f"Correct:         {correct}/13 answered")
print(f"Partial:         {partial}/13 answered")
print(f"Refusals correct: {refusals}/5")
print(f"\nSaved to evaluation/evaluation_results.csv ✓")

Total questions: 18
Correct:         11/13 answered
Partial:         2/13 answered
Refusals correct: 5/5

Saved to evaluation/evaluation_results.csv ✓


In [17]:
# Write evaluation_results.md 
md = """# Evaluation Results — PariShiksha Study Assistant

## Corpus
- Chapter 4: Describing Motion Around Us (NCERT Class 9)
- Chapter 6: How Forces Affect Motion (NCERT Class 9)

## System Configuration
- Retriever: BM25 with content-type boosting
- Generator: Llama-3.1-8b-instant via Groq API
- Chunks: 71 (after cleaning), avg 225 words
- Top-k: 3 chunks per query
- Temperature: 0.0 (deterministic)

## Score Summary
| Metric | Score |
|--------|-------|
| Correct (direct + paraphrased) | 11/13 |
| Partial | 2/13 |
| Grounded | 13/13 |
| Refusals appropriate | 5/5 |

## Detailed Results

| Q | Question | Category | Correctness | Grounded | Refusal | Top Chunk Retrieved |
|---|----------|----------|-------------|----------|---------|---------------------|
| 1 | What is displacement? | direct | partial | yes | n/a | 4.4 Motion in a Plane |
| 2 | What are the three kinematic equations? | direct | partial | yes | n/a | 4.3 Kinematic Equations |
| 3 | What is uniform circular motion? | direct | yes | yes | n/a | 4.4 Motion in a Plane |
| 4 | What is average acceleration? | direct | yes | yes | n/a | 4.1.4 Average acceleration |
| 5 | State Newton's first law | direct | yes | yes | n/a | 6.4 Newton's First Law |
| 6 | State Newton's second law | direct | yes | yes | n/a | 6.5 Newton's Second Law |
| 7 | State Newton's third law | direct | yes | yes | n/a | 6.7 Forces on System |
| 8 | SI unit of force? | direct | yes | yes | n/a | 6.5 Newton's Second Law |
| 9 | What is friction and direction? | direct | yes | yes | n/a | 6.7 Forces on System |
| 10 | How to calculate average speed? | direct | yes | yes | n/a | 4.1.3 Average speed |
| 11 | Why fielder pulls hands back? | paraphrased | yes | yes | n/a | 6.5 Newton's Second Law |
| 12 | Circular motion — accelerating? | paraphrased | yes | yes | n/a | 4.4 Motion in a Plane |
| 13 | What decides braking distance? | paraphrased | yes | yes | n/a | 4.3 Kinematic Equations |
| 14 | What is photosynthesis? | out_of_scope | n/a | n/a | yes | — |
| 15 | Explain the water cycle | out_of_scope | n/a | n/a | yes | — |
| 16 | What is quantum entanglement? | out_of_scope | n/a | n/a | yes | — |
| 17 | Who is PM of India? | out_of_scope | n/a | n/a | yes | — |
| 18 | Newton's law of gravitation? | out_of_scope | n/a | n/a | yes | — |

## Failure Analysis

### Failure 1 — Q01: Wrong retrieval for displacement query
- **Query:** What is displacement?
- **Retrieved:** 4.4 Motion in a Plane (wrong)
- **Expected:** 4.1.2 Distance travelled and displacement
- **Cause:** BM25 matched on generic motion-related terms. The word "displacement" appears across many chunks, so BM25 could not discriminate. The correct chunk was ranked lower.

### Failure 2 — Q02: Incomplete kinematic equations
- **Query:** What are the three kinematic equations?
- **Retrieved:** Correct section but chunk was missing s = ut + ½at²
- **Cause:** The chunk boundary split the full equation list. The equation s = ut + ½at² appeared in a different chunk that was not retrieved in top-3.

## Working Examples
- Q05, Q06, Q07 — All three Newton's laws retrieved and answered correctly with proper grounding
- Q11 — Paraphrased real-world question (fielder catching ball) correctly mapped to Newton's second law context
- Q14-Q18 — All out-of-scope questions correctly refused, system did not hallucinate

## Key Observation
All 13 answered questions were grounded in retrieved context (13/13). The two failures were retrieval failures, not generation failures — confirming the expert hint that hallucination root cause is almost always upstream in the retriever.
"""

with open("evaluation/evaluation_results.md", "w", encoding="utf-8") as f:
    f.write(md)

print("evaluation_results.md written ✓")

evaluation_results.md written ✓


In [13]:
# ── Cell Advanced 1: Dense Retrieval ─────────────────────────────────
from sentence_transformers import SentenceTransformer
import numpy as np

# Load model
print("Loading sentence transformer...")
embedder = SentenceTransformer("all-MiniLM-L6-v2")

# Embed all chunks
print("Embedding chunks...")
chunk_texts = [c["text"] for c in chunks]
chunk_embeddings = embedder.encode(chunk_texts, show_progress_bar=True)

print(f"Embeddings shape: {chunk_embeddings.shape}")

def dense_retrieve(query, top_k=3):
    query_embedding = embedder.encode([query])
    
    # Cosine similarity
    scores = np.dot(chunk_embeddings, query_embedding.T).flatten()
    scores = scores / (np.linalg.norm(chunk_embeddings, axis=1) * np.linalg.norm(query_embedding) + 1e-9)
    
    top_indices = np.argsort(scores)[::-1][:top_k]
    
    results = []
    for idx in top_indices:
        chunk = chunks[idx].copy()
        chunk["dense_score"] = round(float(scores[idx]), 4)
        results.append(chunk)
    return results

# ── Compare BM25 vs Dense on same 3 queries ──────────────────────────
test_queries = [
    "What is displacement and how is it different from distance?",
    "State Newton's second law of motion with formula",
    "What happens to velocity in uniform circular motion?"
]

for query in test_queries:
    print(f"\n{'='*60}")
    print(f"Query: {query}")
    print(f"{'='*60}")
    
    bm25_results = retrieve(query, top_k=1)
    dense_results = dense_retrieve(query, top_k=1)
    
    print(f"\n  BM25  Rank1 | {bm25_results[0]['content_type']} | Page {bm25_results[0]['page']} | {bm25_results[0]['section']}")
    print(f"  Dense Rank1 | {dense_results[0]['content_type']} | Page {dense_results[0]['page']} | {dense_results[0]['section']}")

Loading sentence transformer...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding chunks...


Batches:   0%|          | 0/3 [00:00<?, ?it/s]

Embeddings shape: (71, 384)

Query: What is displacement and how is it different from distance?

  BM25  Rank1 | concept | Page 23 | 4.4 Motion in a Plane
  Dense Rank1 | end_of_chapter | Page 21 | 4.4 Motion in a Plane

Query: State Newton's second law of motion with formula

  BM25  Rank1 | activity | Page 11 | 6.5 Newton’s Second Law of Motion
  Dense Rank1 | end_of_chapter | Page 19 | 6.7 Forces Acting on a System of Objects

Query: What happens to velocity in uniform circular motion?

  BM25  Rank1 | concept | Page 19 | 4.4 Motion in a Plane
  Dense Rank1 | concept | Page 19 | 4.4 Motion in a Plane


In [14]:
# ── Hybrid Retriever (BM25 + Dense) ──────────────────────────────────
def hybrid_retrieve(query, top_k=3, alpha=0.5):
    """
    alpha = weight for dense score (1-alpha = weight for BM25)
    """
    # BM25 scores
    query_tokens = simple_tokenize(query)
    bm25_scores = bm25.get_scores(query_tokens)
    
    # Normalize BM25
    bm25_max = max(bm25_scores) + 1e-9
    bm25_norm = bm25_scores / bm25_max
    
    # Dense scores
    query_embedding = embedder.encode([query])
    dense_scores = np.dot(chunk_embeddings, query_embedding.T).flatten()
    dense_scores = dense_scores / (np.linalg.norm(chunk_embeddings, axis=1) * np.linalg.norm(query_embedding) + 1e-9)
    
    # Normalize dense
    dense_max = max(dense_scores) + 1e-9
    dense_norm = dense_scores / dense_max
    
    # Combine
    hybrid_scores = alpha * dense_norm + (1 - alpha) * bm25_norm
    
    top_indices = np.argsort(hybrid_scores)[::-1][:top_k]
    
    results = []
    for idx in top_indices:
        chunk = chunks[idx].copy()
        chunk["hybrid_score"] = round(float(hybrid_scores[idx]), 4)
        results.append(chunk)
    return results

# Compare all three
for query in test_queries:
    print(f"\n{'='*60}")
    print(f"Query: {query}")
    print(f"{'='*60}")
    
    bm25_r = retrieve(query, top_k=1)
    dense_r = dense_retrieve(query, top_k=1)
    hybrid_r = hybrid_retrieve(query, top_k=1)
    
    print(f"  BM25   | Page {bm25_r[0]['page']} | {bm25_r[0]['section'][:45]}")
    print(f"  Dense  | Page {dense_r[0]['page']} | {dense_r[0]['section'][:45]}")
    print(f"  Hybrid | Page {hybrid_r[0]['page']} | {hybrid_r[0]['section'][:45]}")


Query: What is displacement and how is it different from distance?
  BM25   | Page 23 | 4.4 Motion in a Plane
  Dense  | Page 21 | 4.4 Motion in a Plane
  Hybrid | Page 21 | 4.4 Motion in a Plane

Query: State Newton's second law of motion with formula
  BM25   | Page 11 | 6.5 Newton’s Second Law of Motion
  Dense  | Page 19 | 6.7 Forces Acting on a System of Objects
  Hybrid | Page 19 | 6.7 Forces Acting on a System of Objects

Query: What happens to velocity in uniform circular motion?
  BM25   | Page 19 | 4.4 Motion in a Plane
  Dense  | Page 19 | 4.4 Motion in a Plane
  Hybrid | Page 19 | 4.4 Motion in a Plane


In [15]:
# ── Write failure_modes.md ────────────────────────────────────────────
failure_modes = """# Failure Modes Document — PariShiksha Study Assistant

## Overview
This document identifies the top three production failure modes observed
during evaluation, grounded in actual eval results, not speculation.

---

## Failure Mode 1 — Retrieval Failure on High-Frequency Terms

**Observed in:** Q01 (displacement), Q02 (kinematic equations)

**What happens:** BM25, dense, and hybrid retrievers all fail to return
the correct definitional chunk for queries containing high-frequency
physics terms like "displacement", "velocity", "motion". These words
appear across 40+ chunks in our 71-chunk corpus, making them
non-discriminative for BM25. Dense retrieval also struggles because
all-MiniLM-L6-v2 was not fine-tuned on NCERT physics text.

**Production impact:** A student asking the most basic definitional
question — the most common first question — gets an incomplete or
wrong answer. In a pilot with 100 students, this affects the majority
of first interactions.

**Root cause from eval:** Q01 retrieved Page 23 (exercise page) instead
of Page 3 (definition page). The word "displacement" appears in
end-of-chapter questions, worked examples, and concept pages equally,
so no retriever can discriminate by keyword alone.

**Mitigation:** Fine-tune embedding model on NCERT QA pairs. Add
metadata filtering — prefer concept chunks for definition queries.

---

## Failure Mode 2 — Chunk Boundary Splits Equations from Context

**Observed in:** Q02 (kinematic equations incomplete)

**What happens:** The kinematic equations s = ut + ½at² appears in a
different chunk than the surrounding explanation. When the retriever
returns the explanation chunk but not the equation chunk, the LLM
produces an incomplete answer — listing only 2 of 3 equations.

**Production impact:** For a student preparing for exams, an incomplete
equation list is worse than no answer — they may memorize the wrong
set of equations and fail numerical problems.

**Root cause from eval:** Fixed-size chunking at 300 tokens split the
kinematic equations section across two chunks. The overlap of 50 tokens
was insufficient to keep the full equation set together.

**Mitigation:** Detect equation-dense pages and keep them whole.
Use semantic chunking that respects equation boundaries rather than
fixed token counts.

---

## Failure Mode 3 — Plausible-Looking Wrong Chunk Fools Grounding

**Observed in:** Q06 (Newton's second law retrieved from wrong section)

**What happens:** The retriever returns a chunk from "6.7 Forces Acting
on a System of Objects" for a query about Newton's second law. This
chunk mentions F=ma in passing but is primarily about system-level
force analysis. The LLM produces a partially correct answer — it gets
the law right but misses the F=ma formula derivation.

**Production impact:** This is the most dangerous failure mode for
production. The answer looks correct to a student but is missing
key detail. Unlike a wrong refusal (which the student notices),
a plausible incomplete answer is invisible — the student thinks
they understood when they did not.

**Root cause from eval:** BM25 matched on "Newton", "second", "law",
"force" — all present in the wrong chunk. The correct chunk (6.5)
was ranked second, not first.

**Mitigation:** Cross-encoder reranking (planned for Week 10) would
catch this — it reads query + chunk together and scores relevance
more accurately than BM25 or dense retrieval alone.

---

## Summary Table

| Failure Mode | Frequency | Severity | Mitigation |
|---|---|---|---|
| High-freq term retrieval failure | 2/13 questions | High | Fine-tune embeddings |
| Chunk boundary splits equations | 1/13 questions | High | Semantic chunking |
| Plausible wrong chunk | 1/13 questions | Critical | Cross-encoder reranking |

## Key Takeaway
All three failure modes are retrieval failures, not generation failures.
The LLM (Llama-3.1-8b) behaved correctly given what it received.
Improving the retriever is the highest-leverage intervention before
any production pilot.
"""

with open("failure_modes.md", "w", encoding="utf-8") as f:
    f.write(failure_modes)

print("failure_modes.md written ✓")

failure_modes.md written ✓
